# 07. 三分支融合（FP32）

前面三对 notebook（01/02 Context，03/04 Hybrid，05/06 Delay）各自把一个分支从训练
做到量化。这个 notebook 把三个分支的 **FP32** checkpoint（notebook 01/03/05 的产物）
融合起来，复现 README 里报告的 **91.10%** 测试集结果——方法和教程 04 章一致：
验证集单纯形网格搜索融合权重 + Rest-logit 偏置校准。

量化后的融合（三个分支都换成硬件友好版本）在 [08_fusion_hw_qat.ipynb](08_fusion_hw_qat.ipynb)。

In [ ]:
import sys, importlib.util
from pathlib import Path

import numpy as np
import torch
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import accuracy_score, f1_score

PROJECT_ROOT = Path("training/semg_snn_90_loop")
DELAY_PROJECT_ROOT = Path("training/semg_snn_fpga_reproduction")
sys.path.insert(0, str(PROJECT_ROOT))

from train import EMGDataset
from model import ClassAdaptiveContextSNN, HybridSNN

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

# 三个分支各自 notebook 训练出的 checkpoint;找不到就退回项目自带的已训练结果
def resolve(nb_path: Path, fallback: Path) -> Path:
    return nb_path if nb_path.exists() else fallback

CONTEXT_CHECKPOINT = resolve(
    PROJECT_ROOT / "runs_notebook" / "context23_stage2_stream_nb" / "best.pt",
    PROJECT_ROOT / "runs" / "context23_class_plif_stream" / "best.pt",
)
HYBRID_CHECKPOINT = resolve(
    PROJECT_ROOT / "runs_notebook" / "hybrid_sja_nb" / "best.pt",
    PROJECT_ROOT / "runs" / "hybrid_sja_v1" / "best.pt",
)
DELAY_CHECKPOINT = resolve(
    DELAY_PROJECT_ROOT / "runs_notebook" / "delay62_finetune_nb" / "best.pt",
    DELAY_PROJECT_ROOT / "runs" / "delay62_finetune" / "best.pt",
)
print("Context:", CONTEXT_CHECKPOINT)
print("Hybrid: ", HYBRID_CHECKPOINT)
print("Delay:  ", DELAY_CHECKPOINT)

## 融合方法回顾（教程 04 章）

三个分支各自输出 softmax 概率，加权求和：
`p_fused = w_context*p_context + w_hybrid*p_hybrid + w_delay*p_delay`。
权重在验证集上做单纯形网格搜索（分辨率 10），测试集只用最终选定的权重评估**一次**。
之后再做一次 Rest 类 logit 偏置校准（同样只在验证集上搜）。

In [ ]:
@torch.no_grad()
def predict_context(split: str, checkpoint: Path):
    dataset = EMGDataset(
        PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz",
        False, context=23, continuous_context=True, stream_context=True,
    )
    model = ClassAdaptiveContextSNN(dataset.features.shape[1]).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=False)["model"])
    model.eval()
    outputs = []
    for f, raw, _, subject in DataLoader(dataset, 512, num_workers=4, pin_memory=True):
        logits, _ = model(f.to(device), raw.to(device), subject.to(device))
        outputs.append(logits.softmax(1).cpu().numpy())
    return np.concatenate(outputs), dataset.y


@torch.no_grad()
def predict_hybrid(split: str, checkpoint: Path):
    dataset = EMGDataset(
        PROJECT_ROOT / "data" / f"{split}.npz", PROJECT_ROOT / "data" / "normalization.npz", False, context=1
    )
    model = HybridSNN(dataset.features.shape[1]).to(device)
    model.load_state_dict(torch.load(checkpoint, map_location=device, weights_only=False)["model"])
    model.eval()
    outputs = []
    for f, raw, _, subject in DataLoader(dataset, 512, num_workers=4, pin_memory=True):
        logits, _ = model(f.to(device), raw.to(device), subject.to(device))
        outputs.append(logits.softmax(1).cpu().numpy())
    return np.concatenate(outputs), dataset.y


@torch.no_grad()
def predict_delay(split: str, checkpoint: Path, temperature: float = 0.05):
    spec = importlib.util.spec_from_file_location("paper_snn_model", DELAY_PROJECT_ROOT / "model.py")
    module = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(module)
    state = torch.load(checkpoint, map_location=device, weights_only=False)
    saved_args = state.get("args", {})
    model = module.PaperSNNWithDelays(
        decay=saved_args.get("decay", 0.9), threshold=saved_args.get("threshold", 1.0),
        max_delay=saved_args.get("max_delay", 62), initial_delay=saved_args.get("initial_delay", 1.0),
    ).to(device)
    model.load_state_dict(state["model"])
    model.eval()
    source = np.load(DELAY_PROJECT_ROOT / "data" / "processed" / f"{split}.npz")
    x = torch.from_numpy(source["x"].astype(np.float32))
    y = source["y"].astype(np.int64)
    outputs = []
    for (batch,) in DataLoader(TensorDataset(x), 256, num_workers=4, pin_memory=True):
        spikes, _ = model(batch.to(device))
        outputs.append((spikes.mean(1) / temperature).softmax(1).cpu().numpy())
    return np.concatenate(outputs), y


def score(y, probability):
    p = probability.argmax(1)
    return {
        "accuracy": accuracy_score(y, p),
        "macro_f1": f1_score(y, p, average="macro"),
        "gesture_accuracy": float(np.mean(p[y != 0] == y[y != 0])),
    }

probabilities, labels = {"val": [], "test": []}, {}
for split in probabilities:
    p_context, y1 = predict_context(split, CONTEXT_CHECKPOINT)
    p_hybrid, y2 = predict_hybrid(split, HYBRID_CHECKPOINT)
    p_delay, y3 = predict_delay(split, DELAY_CHECKPOINT)
    assert np.array_equal(y1, y2) and np.array_equal(y1, y3), f"{split} 三个分支的标签顺序没对齐"
    probabilities[split] = [p_context, p_hybrid, p_delay]
    labels[split] = y1
    print(f"{split}: context={score(y1,p_context)} hybrid={score(y2,p_hybrid)} delay={score(y3,p_delay)}")

In [ ]:
# 验证集单纯形网格搜索(分辨率 10 -> 66 组候选权重)
candidates = [(a / 10, b / 10, (10 - a - b) / 10) for a in range(11) for b in range(11 - a)]
rows = [{"weights": w, **score(labels["val"], sum(wi * p for wi, p in zip(w, probabilities["val"])))}
        for w in candidates]
best = max(rows, key=lambda r: (r["accuracy"], r["macro_f1"]))
print("验证集选出的融合权重 (context, hybrid, delay):", best["weights"])

test_probability = sum(w * p for w, p in zip(best["weights"], probabilities["test"]))
test_uncalibrated = score(labels["test"], test_probability)
print("\n测试集(未校准):", test_uncalibrated)
print("参考值(真实项目 strict_stream_three_expert_metrics.json, 权重 0.5/0.4/0.1):")
print("  accuracy=0.9074 macro_f1=0.8195 gesture_accuracy=0.7715")

In [ ]:
# Rest-logit 偏置校准: 只在验证集上搜,缓解 63% Rest 类的系统性偏向
val_probability = sum(w * p for w, p in zip(best["weights"], probabilities["val"]))
val_log_probability = np.log(np.clip(val_probability, 1e-8, 1.0))
bias_rows = []
for rest_bias in np.linspace(-0.8, 0.8, 65):
    calibrated = val_log_probability.copy()
    calibrated[:, 0] += rest_bias
    bias_rows.append({"rest_logit_bias": float(rest_bias), **score(labels["val"], calibrated)})
best_bias = max(bias_rows, key=lambda r: (r["accuracy"], r["macro_f1"]))
print("验证集选出的 Rest 偏置:", best_bias["rest_logit_bias"])

test_log_probability = np.log(np.clip(test_probability, 1e-8, 1.0))
test_log_probability[:, 0] += best_bias["rest_logit_bias"]
test_calibrated = score(labels["test"], test_log_probability)
print("\n=== 最终三分支融合 + 校准后测试集结果 ===")
for k, v in test_calibrated.items():
    print(f"  {k}: {v:.4f}")
print("\n参考值(真实项目, README headline 结果, Rest bias=-0.55):")
print("  accuracy=0.9110  macro_f1=0.8235  gesture_accuracy=0.7977")

## 小结

三个结构完全不同的分支（聚合特征的 Context-SNN、看原始波形的 Hybrid ConvLIF-SNN、
延迟编码的 Delay-SNN）各自单独都到不了 90%（Context ~88.5%、Hybrid ~88.8%、
Delay ~83.9%），融合之后在完全没有动作边界信息、不使用未来数据的最严格在线条件下
做到了 **91.10%**——这正是教程 04 章"多个错误不相关的专家 + 验证集选权重"这套方法论
在真实数据上的效果，也说明**融合收益不是简单地被最强分支拉动的**：即使精度最低的
Delay 分支，只要它的错误模式和另外两个不完全相关，依然能为融合结果做出正贡献。

## 下一步

打开 [08_fusion_hw_qat.ipynb](08_fusion_hw_qat.ipynb)，把三个分支换成量化后的版本
（Context/Hybrid 的 HW-QAT checkpoint + Delay 保持 FP32 概率输出，理由见该 notebook），
重新融合，复现量化后 91.11% 的结果，并做资源估算。